In [1]:
import requests
import pandas as pd

url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
resposta = requests.get(url_ipca)
ipca = pd.DataFrame(resposta.json())

ipca["data"] = pd.to_datetime(ipca["data"], dayfirst=True)
ipca = ipca[ipca["data"] >= "2023-01-01"]

print(ipca)

          data  valor
516 2023-01-01   0.53
517 2023-02-01   0.84
518 2023-03-01   0.71
519 2023-04-01   0.61
520 2023-05-01   0.23
521 2023-06-01  -0.08
522 2023-07-01   0.12
523 2023-08-01   0.23
524 2023-09-01   0.26
525 2023-10-01   0.24
526 2023-11-01   0.28
527 2023-12-01   0.56
528 2024-01-01   0.42
529 2024-02-01   0.83
530 2024-03-01   0.16
531 2024-04-01   0.38
532 2024-05-01   0.46
533 2024-06-01   0.21
534 2024-07-01   0.38
535 2024-08-01  -0.02
536 2024-09-01   0.44
537 2024-10-01   0.56
538 2024-11-01   0.39
539 2024-12-01   0.52
540 2025-01-01   0.16
541 2025-02-01   1.31
542 2025-03-01   0.56
543 2025-04-01   0.43
544 2025-05-01   0.26
545 2025-06-01   0.24
546 2025-07-01   0.26
547 2025-08-01  -0.11
548 2025-09-01   0.48
549 2025-10-01   0.09
550 2025-11-01   0.18
551 2025-12-01   0.33
552 2026-01-01   0.33
553 2026-02-01   0.70
554 2026-03-01   0.88
555 2026-04-01   0.67
556 2026-05-01   0.58
557 2026-06-01   0.16


In [2]:
url_expectativas = (
    "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/"
    "ExpectativaMercadoMensais?$top=10000&$format=json"
    "&$filter=Indicador%20eq%20%27IPCA%27%20and%20Data%20ge%20%272023-01-01%27"
    "&$orderby=Data%20asc"
)

resposta = requests.get(url_expectativas)
print(f"Status code: {resposta.status_code}")

expectativas = pd.DataFrame(resposta.json()["value"])
print(f"Total de linhas: {len(expectativas)}")
print(f"Data mínima: {expectativas['Data'].min()}")
print(f"Data máxima: {expectativas['Data'].max()}")

Status code: 200
Total de linhas: 10000
Data mínima: 2023-01-02
Data máxima: 2023-10-18


In [3]:
expectativas["Data"] = pd.to_datetime(expectativas["Data"])
expectativas["DataReferencia"] = pd.to_datetime(expectativas["DataReferencia"], format="%m/%Y")

expectativas = expectativas.sort_values("Data")
consenso_ipca = expectativas.groupby("DataReferencia").last().reset_index()

print(consenso_ipca[["DataReferencia", "Data", "Mediana"]])

   DataReferencia       Data  Mediana
0      2022-12-01 2023-01-09   0.4600
1      2023-01-01 2023-02-08   0.5550
2      2023-02-01 2023-03-09   0.7800
3      2023-03-01 2023-04-10   0.7700
4      2023-04-01 2023-05-11   0.5500
5      2023-05-01 2023-06-06   0.3700
6      2023-06-01 2023-07-10  -0.1000
7      2023-07-01 2023-08-10   0.0600
8      2023-08-01 2023-09-11   0.2600
9      2023-09-01 2023-10-10   0.3500
10     2023-10-01 2023-10-18   0.3500
11     2023-11-01 2023-10-18   0.3256
12     2023-12-01 2023-10-18   0.5200
13     2024-01-01 2023-10-18   0.4200
14     2024-02-01 2023-10-18   0.5000
15     2024-03-01 2023-10-18   0.3350
16     2024-04-01 2023-10-18   0.3500
17     2024-05-01 2023-10-18   0.2600
18     2024-06-01 2023-10-18   0.2100
19     2024-07-01 2023-10-18   0.2000
20     2024-08-01 2023-10-18   0.1600
21     2024-09-01 2023-10-18   0.2300
22     2024-10-01 2023-10-18   0.3200
23     2024-11-01 2023-10-18   0.3000
24     2024-12-01 2023-10-18   0.4500
25     2025-

In [4]:
ipca_realizado = ipca.rename(columns={"data": "DataReferencia", "valor": "actual"})
ipca_realizado["DataReferencia"] = pd.to_datetime(ipca_realizado["DataReferencia"])
ipca_realizado["actual"] = ipca_realizado["actual"].astype(float)

eventos_ipca = ipca_realizado.merge(
    consenso_ipca[["DataReferencia", "Mediana"]],
    on="DataReferencia",
    how="inner"
)
eventos_ipca = eventos_ipca.rename(columns={"Mediana": "forecast"})

print(eventos_ipca)

   DataReferencia  actual  forecast
0      2023-01-01    0.53    0.5550
1      2023-02-01    0.84    0.7800
2      2023-03-01    0.71    0.7700
3      2023-04-01    0.61    0.5500
4      2023-05-01    0.23    0.3700
5      2023-06-01   -0.08   -0.1000
6      2023-07-01    0.12    0.0600
7      2023-08-01    0.23    0.2600
8      2023-09-01    0.26    0.3500
9      2023-10-01    0.24    0.3500
10     2023-11-01    0.28    0.3256
11     2023-12-01    0.56    0.5200
12     2024-01-01    0.42    0.4200
13     2024-02-01    0.83    0.5000
14     2024-03-01    0.16    0.3350
15     2024-04-01    0.38    0.3500
16     2024-05-01    0.46    0.2600
17     2024-06-01    0.21    0.2100
18     2024-07-01    0.38    0.2000
19     2024-08-01   -0.02    0.1600
20     2024-09-01    0.44    0.2300
21     2024-10-01    0.56    0.3200
22     2024-11-01    0.39    0.3000
23     2024-12-01    0.52    0.4500
24     2025-01-01    0.16    0.4200
25     2025-02-01    1.31    0.4310
26     2025-03-01    0.56   

In [5]:
eventos_ipca["indicador"] = "IPCA_BR"
eventos_ipca["data"] = eventos_ipca["DataReferencia"].dt.strftime("%Y-%m-%d")

eventos_ipca_final = eventos_ipca[["indicador", "data", "actual", "forecast"]]
print(eventos_ipca_final)

   indicador        data  actual  forecast
0    IPCA_BR  2023-01-01    0.53    0.5550
1    IPCA_BR  2023-02-01    0.84    0.7800
2    IPCA_BR  2023-03-01    0.71    0.7700
3    IPCA_BR  2023-04-01    0.61    0.5500
4    IPCA_BR  2023-05-01    0.23    0.3700
5    IPCA_BR  2023-06-01   -0.08   -0.1000
6    IPCA_BR  2023-07-01    0.12    0.0600
7    IPCA_BR  2023-08-01    0.23    0.2600
8    IPCA_BR  2023-09-01    0.26    0.3500
9    IPCA_BR  2023-10-01    0.24    0.3500
10   IPCA_BR  2023-11-01    0.28    0.3256
11   IPCA_BR  2023-12-01    0.56    0.5200
12   IPCA_BR  2024-01-01    0.42    0.4200
13   IPCA_BR  2024-02-01    0.83    0.5000
14   IPCA_BR  2024-03-01    0.16    0.3350
15   IPCA_BR  2024-04-01    0.38    0.3500
16   IPCA_BR  2024-05-01    0.46    0.2600
17   IPCA_BR  2024-06-01    0.21    0.2100
18   IPCA_BR  2024-07-01    0.38    0.2000
19   IPCA_BR  2024-08-01   -0.02    0.1600
20   IPCA_BR  2024-09-01    0.44    0.2300
21   IPCA_BR  2024-10-01    0.56    0.3200
22   IPCA_B

In [7]:
import yfinance as yf

ewz = yf.download("EWZ", start="2023-01-01", end="2026-08-06", progress=True)
ewz = ewz[["Close"]].reset_index()
ewz.columns = ["data", "close"]
ewz["data"] = ewz["data"].dt.strftime("%Y-%m-%d")
ewz["retorno_pct"] = ewz["close"].pct_change() * 100

ewz.to_csv("../data/ewz_precos.csv", index=False)
print(ewz.tail())

[*********************100%***********************]  1 of 1 completed

           data      close  retorno_pct
895  2026-07-30  36.529999     2.988434
896  2026-07-31  36.650002     0.328505
897  2026-08-03  36.419998    -0.627567
898  2026-08-04  36.090000    -0.906090
899  2026-08-05  36.110001     0.055418


In [8]:
import pandas as pd

eventos_principal = pd.read_csv("../data/eventos.csv")
eventos_atualizado = pd.concat([eventos_principal, eventos_ipca_final], ignore_index=True)

eventos_atualizado.to_csv("../data/eventos.csv", index=False)
print(eventos_atualizado["indicador"].value_counts())

indicador
IPCA_BR    68
CPI_EUA    39
Name: count, dtype: int64


In [13]:
eventos_atualizado = eventos_atualizado.drop_duplicates()
print(eventos_atualizado["indicador"].value_counts())

eventos_atualizado.to_csv("../data/eventos.csv", index=False)

indicador
CPI_EUA    39
IPCA_BR    34
Name: count, dtype: int64


In [14]:
import sys
sys.path.append("..")
from src.features import calcular_surpresa, calcular_ian, calcular_ice

eventos_ipca_completo = eventos_atualizado[eventos_atualizado["indicador"] == "IPCA_BR"].copy()

eventos_ipca_completo = calcular_surpresa(eventos_ipca_completo)
eventos_ipca_completo = calcular_ian(eventos_ipca_completo, termos_busca=["IPCA", "inflação"], geo="BR")
eventos_ipca_completo = calcular_ice(
    eventos_ipca_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

print(eventos_ipca_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-01-01        -0.118954  1.000000  0.409959
1  2023-02-01         0.285489  0.566038  0.386531
2  2023-03-01        -0.285489  0.377358  0.362634
3  2023-04-01         0.285489  0.320755  0.338300
4  2023-05-01        -0.666142  0.264151  0.307314
5  2023-06-01         0.095163  0.245283  0.282118
6  2023-07-01         0.285489  0.301887  0.256604
7  2023-08-01        -0.142745  0.132075  0.224350
8  2023-09-01        -0.428234  0.132075  0.198335
9  2023-10-01        -0.523397  0.018868  0.165723
10 2023-11-01        -0.216972  0.037736  0.139804
11 2023-12-01         0.190326  0.018868  0.114522
12 2024-01-01         0.000000  0.566038  0.603649
13 2024-02-01         1.570191  0.509434  0.045959
14 2024-03-01        -0.832677  0.358491  0.403454
15 2024-04-01         0.142745  0.415094 -0.228505
16 2024-05-01         0.951631  0.188679  0.042577
17 2024-06-01         0.000000  0.000000 -0.194023
18 2024-07-01         0.856468 

In [15]:
eventos_ipca_completo.to_csv("../data/eventos_ipca_completo.csv", index=False)

In [16]:
import pandas as pd

cpi_completo = pd.read_csv("../data/eventos_completo.csv")
ipca_completo = pd.read_csv("../data/eventos_ipca_completo.csv")

eventos_todos = pd.concat([cpi_completo, ipca_completo], ignore_index=True)
eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)

print(eventos_todos["indicador"].value_counts())

indicador
CPI_EUA    39
IPCA_BR    34
Name: count, dtype: int64
